In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
seeds = [42, 67, 99, 70, 73]

datasets = {
    "EchoNext": {
        72475: "runs-echonext",
        32768: "runs-echonext-32k",
        16384: "runs-echonext-16k",
        8192: "runs-echonext-8k",
        4096: "runs-echonext-4k",
        2048: "runs-echonext-2k",
        1024: "runs-echonext-1k",
        512: "runs-echonext-512",
        256: "runs-echonext-256",
    },
    "MIMIC-IV-ECG": {
        78470: "runs-mimic",
        32768: "runs-mimic-32k",
        16384: "runs-mimic-16k",
        8192: "runs-mimic-8k",
        4096: "runs-mimic-4k",
        2048: "runs-mimic-2k",
        1024: "runs-mimic-1k",
        512: "runs-mimic-512",
        256: "runs-mimic-256",
    },
    "CODE-15%": {
        74112: "runs-code15",
        32768: "runs-code15-32k",
        16384: "runs-code15-16k",
        8192: "runs-code15-8k",
        4096: "runs-code15-4k",
        2048: "runs-code15-2k",
        1024: "runs-code15-1k",
        512: "runs-code15-512",
        256: "runs-code15-256",
    },
    "PTB-XL": {
        17418: "runs-ptbxl",
        8722: "runs-ptbxl-8k",
        4356: "runs-ptbxl-4k",
        2175: "runs-ptbxl-2k",
        1091: "runs-ptbxl-1k",
        547: "runs-ptbxl-512",
        273: "runs-ptbxl-256",
    },
    "CinC Georgia": {
        8192: "runs-cinc",
        4096: "runs-cinc-4k",
        2048: "runs-cinc-2k",
        1024: "runs-cinc-1k",
        512: "runs-cinc-512",
        256: "runs-cinc-256",
    },
    "ZZU pECG": {
        8658: "runs-zzu",
        4096: "runs-zzu-4k",
        2048: "runs-zzu-2k",
        1024: "runs-zzu-1k",
        512: "runs-zzu-512",
        256: "runs-zzu-256",
    },
}

# mapping of experiment name to tuple of:
# - plotting color
# - folder name
experiments = {
    "Blackbox Direct":     ("tab:grey",   "blackbox-direct"),
    "ECGFounder":          ("tab:red",    "ecgfounder-logreg"),
    "SupProto Direct":     ("tab:green",  "labsup-proto-direct"),
    "ProtoSSL HEEDB":      ("tab:blue",   "protossl-heedb-pila"),
    "ProtoSSL HEEDB (FT)": ("tab:cyan",   "protossl-heedb-pila-ft"),
    "SupProto HEEDB":      ("tab:orange", "labsup-proto-heedb-rila"),
    "SupProto HEEDB (FT)": ("tab:pink", "labsup-proto-heedb-rila-ft"),
    ###
    "ProtoSSL HEEDB (PIA)": ("tab:brown",  "protossl-heedb-pia"),
}


def get_palette(exp_names):
    palette = dict()
    exps = experiments
    for exp_name in exp_names:
        palette[exp_name] = exps[exp_name][0]
    return palette

In [ ]:
data = []
for seed in seeds:
    output_dir = Path(f"/opt/gpu_working/steven/protossl-outputs-seed{seed}")
    for ds, sizes in datasets.items():
        for size, run_dir in sizes.items():
            exps = experiments.copy()
            for exp_name, (exp_color, exp_dir) in exps.items():
                metrics_csv = output_dir / run_dir / exp_dir / "metrics-bootstrapped.csv"
                if not os.path.exists(metrics_csv):
                    continue
                metrics = pd.read_csv(metrics_csv, index_col="Label")
                multilabel = metrics.loc["Multilabel Averaged"]
                datum = {
                    "Seed": seed,
                    "Dataset": ds,
                    "Model": exp_name,
                    "Train Size": size,
                    "Multilabel (AUROC)": multilabel["AUROC"],
                    "Multilabel (AUPRC)": multilabel["AUPRC"],
                }
                if "AUROC 95% CI (lo)" in metrics.columns:
                    datum["AUROC 95% CI (lo)"] = metrics.loc["Multilabel Averaged", "AUROC 95% CI (lo)"]
                    datum["AUROC 95% CI (hi)"] = metrics.loc["Multilabel Averaged", "AUROC 95% CI (hi)"]
                data.append(datum)
results = pd.DataFrame.from_records(data)

In [ ]:
full_scale_agg = (
    results
    .sort_values(["Model", "Dataset", "Train Size"])
    .drop_duplicates(["Model", "Dataset", "Seed"], keep="last")
    .groupby(["Dataset", "Model", "Train Size"])
    # ["Multilabel (AUROC)"].describe() # reporting std dev
    [["Multilabel (AUROC)", "AUROC 95% CI (lo)", "AUROC 95% CI (hi)"]].mean() # reporting bootstrapped CI
)
# full_scale_agg = full_scale_agg.reset_index()[["Dataset", "Model", "mean", "std"]] # reporting std dev
full_scale_agg = full_scale_agg.reset_index()[["Dataset", "Model", "Multilabel (AUROC)", "AUROC 95% CI (lo)", "AUROC 95% CI (hi)"]] # reporting bootstrapped CI

no_ft_full = full_scale_agg[
    full_scale_agg["Model"].isin([
        "SupProto Direct",
        "SupProto HEEDB",
        "SupProto HEEDB (FT)",
        "ProtoSSL HEEDB",
        "ProtoSSL HEEDB (FT)",
    ])
].reset_index(drop=True)

# reporting std dev
# no_ft_full["val_mean"] = no_ft_full["mean"].apply(lambda x: f"{x:0.3f}")
# no_ft_full["val_std"] = no_ft_full["std"].apply(lambda x: f"{x:.2e}")
# no_ft_full["val"] = no_ft_full["val_mean"] + " ± " + no_ft_full["val_std"]

# report bootstrapped CI
no_ft_full["val"] = (
    no_ft_full["Multilabel (AUROC)"].apply(lambda x: f"{x:0.3f}")
    + " ["
    + no_ft_full["AUROC 95% CI (lo)"].apply(lambda x: f"{x:0.3f}")
    + "-"
    + no_ft_full["AUROC 95% CI (hi)"].apply(lambda x: f"{x:0.3f}")
    + "]"
)

pivoted = no_ft_full[["Dataset", "Model", "val"]].pivot(columns="Dataset", index="Model", values="val")
# pivoted.columns.name = None
pivoted.index.name = None
# pivoted = pivoted.reset_index()
pivoted = pivoted.loc[
    ["ProtoSSL HEEDB (FT)", "ProtoSSL HEEDB", "SupProto HEEDB (FT)", "SupProto HEEDB", "SupProto Direct"],
    ["EchoNext", "MIMIC-IV-ECG", "ZZU pECG", "PTB-XL", "CinC Georgia", "CODE-15%"]
]
pivoted = pivoted.T.reset_index()
pivoted = pd.concat([pd.DataFrame({"Domain": ["Out-of-Domain"]*3+["In-Domain"]*3}), pivoted], axis=1)
pivoted = pivoted.set_index(["Domain", "Dataset"])
pivoted.columns = pd.MultiIndex.from_arrays([
    ["ProtoSSL HEEDB", "ProtoSSL HEEDB", "SupProto HEEDB", "SupProto HEEDB", "SupProto Direct"],
    ["Tuned", "Probed", "Tuned", "Probed", ""],
])

print(pivoted.to_latex())

In [ ]:
def plot_lift(
    *,  # enforce kwargs
    df: pd.DataFrame, # long format df (each point to plot is a row)
    dataset: str,
    seed: int | list[int] = 42, # if list of int, average across seeds
    metric: str,
    models: list[str], # must be intentional about which models to plot
    rename: list[str] | None = None,
    baseline_model: str | None = None, # singular result to optionally plot as dashed line
    ylim: tuple[float, float] | None = None,
    xlim: tuple[float, float] | None = None,
    save_path: str | None = None,
    smooth: bool = True,
    ax: plt.Axes | None = None,
    do_label: bool = True,
):
    if rename is not None:
        assert len(models) == len(rename)
    if baseline_model is not None and baseline_model not in models:
        models = [baseline_model] + models
        if rename is not None:
            rename = [baseline_model] + rename
    palette = get_palette(models)
    if rename is not None:
        palette = {new_name: palette[m] for new_name, m in zip(rename, models)}
        df = df.copy()
        df["Model"] = df["Model"].replace({v: k for k, v in zip(rename, models)})

    df = df[df["Dataset"] == dataset]
    if isinstance(seed, int):
        df = df[df["Seed"] == seed]
    else: # list of seeds
        df = df[df["Seed"].isin(seed)]
        df = df.groupby(["Dataset", "Model", "Train Size"]).mean().reset_index()
    fig = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    min_size = df["Train Size"].min()
    max_size = df["Train Size"].max()
    if xlim is not None:
        min_size = min(min_size, xlim[0])
        max_size = max(max_size, xlim[1])

    if baseline_model is not None:
        mask = df["Model"] == baseline_model
        assert (
            mask.sum() == 1
        ), f"Should only have 1 entry for baseline model: {baseline_model}"
        baseline_row = df[mask].iloc[0]
        ax.hlines(
            baseline_row[metric],
            min_size,
            max_size,
            colors=palette.pop(baseline_model),
            linestyles=":",
            label=baseline_model,
        )
        df = df[~mask] # subsequent line plots should exclude baseline model

    if not smooth:
        sns.lineplot(
            df,
            x="Train Size",
            y=metric,
            hue="Model",
            palette=palette,
            hue_order=list(palette.keys()),
            marker="o",
            ax=ax,
        )
    else:
        for model, color in palette.items():
            model_data = df[df["Model"] == model]
            ax.set_xlim([2**7, 2**17])
            sns.regplot(
                model_data,
                x="Train Size",
                y=metric,
                color=color,
                marker="o",
                ax=ax,
                logx=True,
                label=model if do_label else None,
                line_kws={"zorder": 2},
                scatter_kws={"zorder": 3},
                truncate=False,
            )
    ax.set_xscale("log", base=2)
    if xlim is not None:
        ax.set_xlim(xlim)
    else:
        # tighter boundaries than default lims
        ax.set_xlim((min_size, max_size))
    ax.set_title(dataset)
    # ax.set_title(f"{dataset} {metric}")
    if do_label:
        ax.legend(loc="lower right")
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        ymin, ymax = ax.get_ylim()
        ax.set_ylim((ymin, min(ymax, 1)))
    if save_path is not None:
        assert fig is not None
        fig.tight_layout()
        fig.savefig(save_path)

In [ ]:
import string
from matplotlib.lines import Line2D

def make_paper_fig(models):

    fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(5.4*2, 3.35*2))
    for i, (ds, short) in enumerate([
        ("EchoNext", "echonext"),
        ("MIMIC-IV-ECG", "mimic"),
        ("ZZU pECG", "zzu"),
        ("PTB-XL", "ptbxl"),
        ("CinC Georgia", "cinc"),
        ("CODE-15%", "code15"),
    ]):
        ax = axs[i//3, i%3]
        ax.annotate(
            f"({string.ascii_lowercase[i]})",
            xy=(0, 1.02),
            xycoords='axes fraction',
            fontsize=16,
            va='bottom',
            ha='left',
        )
        plot_lift(
            df=results,
            dataset=ds,
            metric="Multilabel (AUROC)",
            models=models,
            ax=ax,
            seed=seeds,
            do_label=i==0
        )

        if short == "echonext":
            ax.set_ylim([0.65, 0.83])
        if short == "mimic":
            ax.set_ylim([0.63, 0.83])
        if short == "zzu":
            ax.set_ylim([0.63, 0.83])
        if short == "ptbxl":
            ax.set_ylim([0.60, 0.94])
        if short == "cinc":
            ax.set_ylim([0.61, 0.91])
        if short == "code15":
            ax.set_ylim([0.67, 1])
            ax.set_xlim([2**8, ax.get_xlim()[1]])
        ax.set_title(ds, fontsize=12, fontweight="bold")
        ax.set_ylabel("")
        if i % 3 == 0:
            if i == 0:
                domain_text = r'$\bf{No\ Overlap\ with\ HEEDB}$'
            else:
                domain_text = r'$\bf{Label\ Overlap\ with\ HEEDB}$'
            ax.set_ylabel(domain_text + '\n───────────────────────────\nMacro AUROC', fontsize=12)

        ax.set_xlabel("")
        ax.text(0.5, 0.0, 'Train Set Size',
                    ha='center', va='bottom',
                    transform=ax.transAxes)
        # ax.set_xlabel("Train Size", fontsize=12)
        ymin, ymax = ax.get_ylim()
        yticks = [y for y in np.arange(0.5, 1.1, 0.1) if y > ymin and y <= ymax]
        ax.set_yticks(yticks)
        ax.tick_params(axis='both', labelsize=12)
        # ax.set_axisbelow(True)
        ax.grid(axis="y")
    leg = axs[0, 0].get_legend()
    new_handles = [
        Line2D([0], [0], marker='o', color='w',
            markerfacecolor=h.get_facecolor()[0],
            markersize=10,
            label=t.get_text())
        for h, t in zip(leg.legend_handles, leg.get_texts())
    ]
    leg.remove()
    fig.legend(
        handles=new_handles,
        loc='lower center', # anchor point on the legend itself
        bbox_to_anchor=(0.5, -0.06), # center-bottom of the figure
        ncols=5, # one column per item = flat row
        frameon=True,
        prop={"weight": "bold", "size": 12},
        handletextpad=0.02, # space between handle and label (default 0.8)
        columnspacing=0.5,
        handleheight=1.25,
    )
    fig.tight_layout(pad=0.1)
    return fig

In [ ]:
fig = make_paper_fig([
    "ProtoSSL HEEDB",
    "ProtoSSL HEEDB (FT)",
    "SupProto HEEDB",
    "SupProto HEEDB (FT)",
    "SupProto Direct",
])
fig.savefig("figs/ecg-efficiency.png", dpi=300, bbox_inches='tight')
fig.savefig("figs/ecg-efficiency.pdf", bbox_inches='tight')

In [ ]:
fig = make_paper_fig([
    "ProtoSSL HEEDB",
    "SupProto HEEDB",
    "Blackbox Direct",
    "ECGFounder",
    # "ST-MEM", # TODO
])
fig.savefig("figs/fm-ecg.png", dpi=300, bbox_inches='tight')
fig.savefig("figs/fm-ecg.pdf", bbox_inches='tight')

In [ ]:
# average across seeds, linear probe top row, fine tune bottom row

fig, axs = plt.subplots(nrows=2, ncols=6, figsize=(36, 12))
for col in range(6):
    axs[1, col].sharey(axs[0, col])
for col, (ds, short) in enumerate([
    ("EchoNext", "echonext"),
    ("MIMIC-IV-ECG (ED)", "mimic"),
    ("PTB-XL", "ptbxl"),
    ("CinC Georgia", "cinc"),
    ("ZZU pediatric ECG", "zzu"),
    ("CODE-15%", "code15"),
]):
    plot_lift(
        df=results,
        dataset=ds,
        metric="Multilabel (AUROC)",
        models=[
            "Blackbox Direct",
            "LabSup Proto Direct",
            "LabSup Proto HEEDB (RILA)",
            "ProtoSSL HEEDB (PILA)",
        ],
        ax=axs[0, col],
        seed=seeds,
    )
    plot_lift(
        df=results,
        dataset=ds,
        metric="Multilabel (AUROC)",
        models=[
            "Blackbox Direct",
            "LabSup Proto Direct",
            "LabSup Proto HEEDB (RILA) (FT)",
            "ProtoSSL HEEDB (PILA) (FT)",
        ],
        ax=axs[1, col],
        seed=seeds,
    )
fig.tight_layout()
fig.savefig("figs/multiseed-avg.png")

In [ ]:
# plot each seed as a separate row

# ==============
# Linear Probe
# ==============

fig, axs = plt.subplots(nrows=5, ncols=6, figsize=(36, 30))
for col in range(6):
    for row in range(1, 5):
        axs[row, col].sharey(axs[0, col])
for row, seed in enumerate(seeds):
    for col, (ds, short) in enumerate([
        ("EchoNext", "echonext"),
        ("MIMIC-IV-ECG (ED)", "mimic"),
        ("PTB-XL", "ptbxl"),
        ("CinC Georgia", "cinc"),
        ("ZZU pediatric ECG", "zzu"),
        ("CODE-15%", "code15"),
    ]):
        plot_lift(
            df=results,
            dataset=ds,
            seed=seed,
            metric="Multilabel (AUROC)",
            models=[
                "Blackbox Direct",
                "LabSup Proto Direct",
                "LabSup Proto HEEDB (RILA)",
                "ProtoSSL HEEDB (PILA)",
            ],
            ax=axs[row, col],
        )
fig.tight_layout()
fig.savefig("figs/multiseed.png")

# ==============
# Fine-Tuned
# ==============

fig, axs = plt.subplots(nrows=5, ncols=6, figsize=(36, 30))
for col in range(6):
    for row in range(1, 5):
        axs[row, col].sharey(axs[0, col])
for row, seed in enumerate(seeds):
    for col, (ds, short) in enumerate([
        ("EchoNext", "echonext"),
        ("MIMIC-IV-ECG (ED)", "mimic"),
        ("PTB-XL", "ptbxl"),
        ("CinC Georgia", "cinc"),
        ("ZZU pediatric ECG", "zzu"),
        ("CODE-15%", "code15"),
    ]):
        plot_lift(
            df=results,
            dataset=ds,
            seed=seed,
            metric="Multilabel (AUROC)",
            models=[
                "Blackbox Direct",
                "LabSup Proto Direct",
                "LabSup Proto HEEDB (RILA) (FT)",
                "ProtoSSL HEEDB (PILA) (FT)",
            ],
            ax=axs[row, col],
        )
fig.tight_layout()
fig.savefig("figs/multiseed-ft.png")

In [ ]:
plot_lift(
    df=results,
    dataset="PTB-XL",
    metric="Multilabel (AUROC)",
    seed=42,
    models=[
        "Blackbox Direct",
        "LabSup Proto Direct",
        "LabSup Proto HEEDB (RILA)",
        "ProtoSSL HEEDB (PILA)",
        "ProtoSSL HEEDB (PIA)",
    ],
    # save_path=f"figs/probe-{short}.png",
)

In [ ]:
# individual files, single seed

# for ds, short in [
#     ("EchoNext", "echonext"),
#     ("MIMIC-IV-ECG (ED)", "mimic"),
#     ("PTB-XL", "ptbxl"),
#     ("CinC Georgia", "cinc"),
#     ("ZZU pediatric ECG", "zzu"),
#     ("CODE-15%", "code15"),
# ]:
#     plot_lift(
#         df=results,
#         dataset=ds,
#         metric="Multilabel (AUROC)",
#         models=[
#             "Blackbox Direct",
#             "LabSup Proto Direct",
#             "LabSup Proto HEEDB (RILA)",
#             "ProtoSSL HEEDB (PILA)",
#         ],
#         save_path=f"figs/probe-{short}.png",
#     )
#     plot_lift(
#         df=results,
#         dataset=ds,
#         metric="Multilabel (AUROC)",
#         models=[
#             "Blackbox Direct",
#             "LabSup Proto Direct",
#             "LabSup Proto HEEDB (RILA) (FT)",
#             "ProtoSSL HEEDB (PILA) (FT)",
#         ],
#         save_path=f"figs/ft-{short}.png",
#     )